# TriFlowQC Tutorial: Introduction to Qutrit Quantum Computing

This notebook provides a hands-on introduction to qutrit (3-state) quantum computing using TriFlowQC.

## Contents
1. What are Qutrits?
2. Creating and Manipulating Qutrits
3. Gell-Mann Matrices (SU(3) Generators)
4. Building Qutrit Circuits
5. Simulation
6. Entanglement in Qutrits
7. Noise Simulation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# TriFlowQC imports
from triflowqc.core.qutrit import Qutrit
from triflowqc.core.register import QutritRegister
from triflowqc.core.density_matrix import DensityMatrix
from triflowqc.gates.gell_mann import gell_mann_matrices
from triflowqc.gates.single_qutrit import X3, Z3, H3
from triflowqc.gates.parametric import RX3, RY3, RZ3
from triflowqc.gates.two_qutrit import CSUM, SWAP3
from triflowqc.circuit.circuit import QutritCircuit
from triflowqc.simulator.statevector_sim import StatevectorSimulator
from triflowqc.simulator.density_sim import DensityMatrixSimulator
from triflowqc.simulator.noise_model import NoiseModel, depolarising_channel
from triflowqc.simulator.noisy_simulator import NoisySimulator
from triflowqc.math.information import von_neumann_entropy, purity

print('TriFlowQC loaded successfully!')

## 1. What are Qutrits?

A **qutrit** is the 3-dimensional generalization of a qubit. While a qubit lives in $\mathbb{C}^2$, a qutrit lives in $\mathbb{C}^3$:

$$|\psi\rangle = \alpha|0\rangle + \beta|1\rangle + \gamma|2\rangle$$

where $|\alpha|^2 + |\beta|^2 + |\gamma|^2 = 1$.

Key advantages of qutrits:
- **Higher information density**: $\log_2(3) \approx 1.585$ bits per qutrit
- **Richer entanglement**: More complex entanglement structures
- **Better error correction**: Certain codes are more efficient in higher dimensions

In [ ]:
# Create qutrits in different states
q0 = Qutrit()  # |0⟩
q1 = Qutrit(state=np.array([0, 1, 0], dtype=complex))  # |1⟩
q2 = Qutrit(state=np.array([0, 0, 1], dtype=complex))  # |2⟩
q_sup = Qutrit(state=np.array([1, 1, 1], dtype=complex) / np.sqrt(3))  # Superposition

print('|0⟩ probabilities:', np.abs(q0.state)**2)
print('|1⟩ probabilities:', np.abs(q1.state)**2)
print('|2⟩ probabilities:', np.abs(q2.state)**2)
print('Superposition probabilities:', np.abs(q_sup.state)**2)

## 2. Single-Qutrit Gates

The fundamental single-qutrit gates are:

- **X3 (Shift)**: $|k\rangle \to |k+1 \mod 3\rangle$
- **Z3 (Clock)**: $|k\rangle \to \omega^k|k\rangle$ where $\omega = e^{2\pi i/3}$
- **H3 (QFT)**: $|j\rangle \to \frac{1}{\sqrt{3}}\sum_k \omega^{jk}|k\rangle$

In [ ]:
# X3 gate: cyclic shift
print('X3 gate (shift operator):')
print(X3.round(3))

q = Qutrit()
print('\n|0⟩:', np.abs(q.state)**2)
q.apply_gate(X3)
print('X3|0⟩ = |1⟩:', np.abs(q.state)**2)
q.apply_gate(X3)
print('X3|1⟩ = |2⟩:', np.abs(q.state)**2)
q.apply_gate(X3)
print('X3|2⟩ = |0⟩:', np.abs(q.state)**2)

# Verify X3³ = I
print('\nX3³ = I:', np.allclose(X3 @ X3 @ X3, np.eye(3)))

In [ ]:
# H3 gate: Hadamard / QFT
print('H3 gate (qutrit QFT):')
print(H3.round(4))

q = Qutrit()
q.apply_gate(H3)
print('\nH3|0⟩ probabilities:', np.abs(q.state)**2)
print('All equal to 1/3?', np.allclose(np.abs(q.state)**2, 1/3))

## 3. Gell-Mann Matrices (SU(3) Generators)

The Gell-Mann matrices $\lambda_1, \ldots, \lambda_8$ are the generators of SU(3), analogous to Pauli matrices for SU(2). They satisfy:

$$[\lambda_i, \lambda_j] = 2i \sum_k f_{ijk} \lambda_k$$

$$\text{Tr}(\lambda_i \lambda_j) = 2\delta_{ij}$$

In [ ]:
lambdas = gell_mann_matrices()

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, (ax, lam) in enumerate(zip(axes.flat, lambdas)):
    # Plot real part
    im = ax.imshow(lam.real, cmap='RdBu', vmin=-1, vmax=1)
    ax.set_title(f'λ_{i+1} (real)')
    ax.set_xticks([0, 1, 2])
    ax.set_yticks([0, 1, 2])
    for r in range(3):
        for c in range(3):
            val = lam[r, c]
            text = f'{val.real:.1f}'
            if abs(val.imag) > 1e-10:
                text += f'\n+{val.imag:.1f}i' if val.imag > 0 else f'\n{val.imag:.1f}i'
            ax.text(c, r, text, ha='center', va='center', fontsize=8)
plt.tight_layout()
plt.suptitle('The 8 Gell-Mann Matrices', y=1.02, fontsize=14)
plt.show()

# Verify properties
print('Properties:')
for i, lam in enumerate(lambdas):
    print(f'  λ_{i+1}: Hermitian={np.allclose(lam, lam.conj().T)}, '
          f'Traceless={abs(np.trace(lam)) < 1e-10}')

## 4. Parametric Rotation Gates

Parametric gates are the building blocks for variational qutrit circuits:

$$R_k(\theta) = \exp\left(-i\frac{\theta}{2}\lambda_k\right)$$

Convenience wrappers:
- `RX3(θ)` = $R_1(\theta)$
- `RY3(θ)` = $R_2(\theta)$
- `RZ3(θ)` = $R_3(\theta)$

In [ ]:
# Sweep RX3 angle and plot probabilities
thetas = np.linspace(0, 2*np.pi, 100)
probs = np.zeros((len(thetas), 3))

for i, theta in enumerate(thetas):
    U = RX3(theta)
    psi = U @ np.array([1, 0, 0], dtype=complex)
    probs[i] = np.abs(psi)**2

plt.figure(figsize=(10, 4))
plt.plot(thetas, probs[:, 0], label='P(|0⟩)', linewidth=2)
plt.plot(thetas, probs[:, 1], label='P(|1⟩)', linewidth=2)
plt.plot(thetas, probs[:, 2], label='P(|2⟩)', linewidth=2)
plt.xlabel('θ')
plt.ylabel('Probability')
plt.title('RX3(θ)|0⟩ Probabilities')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 5. Qutrit Circuits and Simulation

Build circuits using `QutritCircuit` and simulate with `StatevectorSimulator`.

In [ ]:
# Build a circuit
qc = QutritCircuit(2)
qc.h(0)
qc.rx(0, np.pi/4)
qc.csum(0, 1)
qc.rz(1, np.pi/3)

print('Circuit:')
print(qc.draw())
print(f'\nQutrits: {qc.n_qutrits}')
print(f'Depth: {qc.depth()}')
print(f'Parameters: {len(qc.parameters)}')

In [ ]:
# Simulate
sim = StatevectorSimulator()
result = sim.run(qc, shots=10000)

# Plot probabilities
states = sorted(result.probabilities.keys())
probs = [result.probabilities[s] for s in states]

plt.figure(figsize=(10, 4))
plt.bar(states, probs, color='steelblue', alpha=0.8)
plt.xlabel('State')
plt.ylabel('Probability')
plt.title('Measurement Probabilities')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 6. Qutrit Entanglement

Create the qutrit analog of a Bell state:

$$|\Phi^+\rangle = \frac{1}{\sqrt{3}}(|00\rangle + |11\rangle + |22\rangle)$$

This state has maximum entanglement for two qutrits (Schmidt rank 3).

In [ ]:
# Create Bell state
qc_bell = QutritCircuit(2)
qc_bell.h(0)
qc_bell.csum(0, 1)

result_bell = sim.run(qc_bell, shots=30000)

print('Qutrit Bell State probabilities:')
for state, prob in sorted(result_bell.probabilities.items()):
    print(f'  |{state}⟩: {prob:.4f}')

# Plot
states = [f'{a}{b}' for a in range(3) for b in range(3)]
probs = [result_bell.probabilities.get(s, 0) for s in states]

colors = ['#e74c3c' if s in ['00', '11', '22'] else '#3498db' for s in states]
plt.figure(figsize=(10, 4))
plt.bar(states, probs, color=colors, alpha=0.8)
plt.axhline(y=1/3, color='red', linestyle='--', alpha=0.5, label='1/3')
plt.xlabel('State |ab⟩')
plt.ylabel('Probability')
plt.title('Qutrit Bell State: (|00⟩ + |11⟩ + |22⟩)/√3')
plt.legend()
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# Verify entanglement via entropy
from triflowqc.math.algebra import partial_trace_mixed
from triflowqc.math.decompositions import schmidt_decomposition, schmidt_rank

# Construct Bell state vector
psi_bell = np.zeros(9, dtype=complex)
psi_bell[0] = psi_bell[4] = psi_bell[8] = 1/np.sqrt(3)
rho_bell = np.outer(psi_bell, psi_bell.conj())

# Partial trace
rho_A = partial_trace_mixed(rho_bell, [1], d=3, n=2)
print(f'Reduced density matrix ρ_A:')
print(rho_A.round(4))
print(f'\nEntanglement entropy: {von_neumann_entropy(rho_A):.4f} (max = ln(3) = {np.log(3):.4f})')
print(f'Purity of ρ_A: {purity(rho_A):.4f} (min = 1/3 = {1/3:.4f})')
print(f'Schmidt rank: {schmidt_rank(psi_bell, 3, 3)}')

coeffs, _, _ = schmidt_decomposition(psi_bell, 3, 3)
print(f'Schmidt coefficients: {coeffs.round(4)}')

## 7. Noise Simulation

Model realistic noise effects:
- **Depolarising**: Randomly replaces state with maximally mixed
- **Dephasing**: Destroys off-diagonal coherences
- **Amplitude damping**: T₁ decay toward ground state

In [ ]:
# Compare noiseless vs noisy Bell state
noise_levels = [0.0, 0.01, 0.05, 0.1, 0.2, 0.5]
p_correlated = []
entropies = []

for p_noise in noise_levels:
    qc_bell = QutritCircuit(2)
    qc_bell.h(0)
    qc_bell.csum(0, 1)
    
    if p_noise == 0:
        noisy_sim = StatevectorSimulator()
    else:
        noise = NoiseModel()
        noise.add_gate_error('H3', depolarising_channel(p_noise))
        noise.add_gate_error('CSUM', depolarising_channel(p_noise, n_qutrits=2))
        noisy_sim = NoisySimulator(noise)
    
    result = noisy_sim.run(qc_bell, shots=10000)
    p_corr = sum(result.probabilities.get(s, 0) for s in ['00', '11', '22'])
    p_correlated.append(p_corr)

plt.figure(figsize=(10, 4))
plt.plot(noise_levels, p_correlated, 'o-', linewidth=2, markersize=8)
plt.axhline(y=1.0, color='green', linestyle='--', alpha=0.5, label='Ideal')
plt.axhline(y=1/3, color='red', linestyle='--', alpha=0.5, label='Random (1/3)')
plt.xlabel('Depolarising Noise Rate')
plt.ylabel('P(correlated states)')
plt.title('Effect of Depolarising Noise on Bell State')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Summary

In this tutorial, you learned:
- **Qutrits** are 3-state quantum systems (SU(3) instead of SU(2))
- **Gell-Mann matrices** are the generators of SU(3), analogous to Pauli matrices
- **Single-qutrit gates** (X3, Z3, H3) and **parametric rotations** (RX3, RY3, RZ3)
- **Two-qutrit gates** (CSUM, SWAP3) enable entanglement
- **Simulation** with statevector, density matrix, and noisy simulators
- **Entanglement** in qutrits is richer (Schmidt rank up to 3)

Next: See the QNN Training notebook for building quantum neural networks with qutrits!